# 02 — Backtest delle strategie (Fase 2)

In questo notebook usiamo il **motore di backtest** per testare alcune strategie di investimento sui prezzi storici e confrontarne le performance.

Concetti chiave:
- una **strategia** trasforma i prezzi in *pesi target* (quanto capitale allocare su ogni asset, giorno per giorno)
- il **motore** simula il portafoglio applicando i costi di transazione e produce la *equity curve*
- le **metriche** (Sharpe, drawdown) sono le stesse della Fase 1

> Usiamo prezzi sintetici deterministici. Con dati reali basta passare al motore un DataFrame di prezzi (indice = date, colonne = ticker).

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = 'iframe'   # renderer robusto per JupyterLab

import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Root progetto:', ROOT)

## 1. Prezzi storici
Generiamo prezzi sintetici per 3 asset. Con dati reali, sostituisci `prices` con i tuoi prezzi di chiusura.

In [ ]:
from src.ingestion.sample_data import make_backtest_prices

prices = make_backtest_prices(days=750)
prices.tail()

## 2. Una singola strategia
Testiamo un incrocio di medie mobili (SMA 20/50) con un costo di transazione di 10 basis point (0,10%) sul turnover.

In [ ]:
from src.backtest import backtest_strategy, SMACrossover

res = backtest_strategy(prices, SMACrossover(fast=20, slow=50), fee_bps=10)
print('Statistiche:')
for k, v in res.summary().items():
    print(f'  {k:>14}: {v:,.4f}')

In [ ]:
from src.backtest.plots import plot_equity_curve

plot_equity_curve(res).show()

## 3. Confronto fra strategie
Mettiamo a confronto Buy & Hold (benchmark), l'incrocio di medie mobili e una strategia momentum.

In [ ]:
from src.backtest import BuyAndHold, SMACrossover, Momentum, compare_strategies

strategie = [BuyAndHold(), SMACrossover(20, 50), Momentum(60)]
results, table = compare_strategies(prices, strategie, fee_bps=10)
table

In [ ]:
from src.backtest.plots import plot_strategy_comparison

plot_strategy_comparison(results).show()

---
## Come leggere i risultati
- **total_return / cagr**: rendimento totale e annuo composto
- **sharpe**: rendimento aggiustato per il rischio (più alto = meglio)
- **max_drawdown**: perdita massima dal picco (più vicino a 0 = meglio)
- **volatility**: oscillazione annua dei rendimenti
- **n_rebalances**: quante volte la strategia ha cambiato allocazione (più alto = più costi)

> Su dati che salgono sempre, Buy & Hold è difficile da battere. Le strategie trend-following brillano in mercati laterali o ribassisti. Prova a cambiare i parametri (`fast`/`slow`, `lookback`) o i prezzi per vedere come cambiano i risultati.

**Prossima fase:** integrare queste funzioni in una dashboard / agente AI (Fase 4).